# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [1]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [2]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/payment"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 3

--- Payment_20260207170548.parquet ---
  Registros: 2
  Colunas: ['raw_value']
  ❌ Não contém dados JSON

--- Payment_20260325043100.parquet ---
  Registros: 10
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']

--- placeholder_payment_20260324.parquet ---
  Registros: 1
  Colunas: ['raw_value']
  ✅ Contém JSON nas colunas: ['raw_value']



import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [3]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema)).select("parsed.*")
        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 2

Comparando: Payment_20260325043100.parquet vs placeholder_payment_20260324.parquet
  ✅ Mesmas colunas (17 colunas)
  ✅ Tipos de dados idênticos

✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!


In [4]:
from pyspark.sql.functions import from_json, schema_of_json, col, explode_outer, posexplode_outer
from pyspark.sql.types import StructType, StringType, ArrayType

def flatten_df(df, sep="_"):
    """
    Achata recursivamente colunas StructType e ArrayType.
    - StructType com apenas a chave 'value': extrai o valor direto (sem sufixo).
    - StructType com múltiplas chaves: expande como pai_filho.
    - ArrayType de StructType: explode as linhas e achata os campos.
    - ArrayType de tipos simples: explode as linhas.
    """
    mudou = True
    while mudou:
        mudou = False
        novas_colunas = []
        for field in df.schema.fields:
            # StructType
            if isinstance(field.dataType, StructType):
                mudou = True
                sub_fields = field.dataType.fields
                # Regra: se só tem a chave "value", extrai direto sem sufixo
                if len(sub_fields) == 1 and sub_fields[0].name.lower() == "value":
                    novas_colunas.append(
                        df[field.name]["value"].alias(field.name)
                    )
                else:
                    for sub in sub_fields:
                        novo_nome = f"{field.name}{sep}{sub.name}"
                        novas_colunas.append(
                            df[field.name][sub.name].alias(novo_nome)
                        )
            # ArrayType
            elif isinstance(field.dataType, ArrayType):
                mudou = True
                elem_type = field.dataType.elementType
                if isinstance(elem_type, StructType):
                    # Explode o array e achata os campos do struct
                    df = df.withColumn(field.name, explode_outer(col(field.name)))
                    for sub in elem_type.fields:
                        novo_nome = f"{field.name}{sep}{sub.name}"
                        novas_colunas.append(
                            df[field.name][sub.name].alias(novo_nome)
                        )
                else:
                    # Array de tipo simples: explode direto
                    df = df.withColumn(field.name, explode_outer(col(field.name)))
                    novas_colunas.append(df[field.name])
            else:
                novas_colunas.append(df[field.name])
        df = df.select(novas_colunas)
    return df

def parse_nested_json(df, sep="_", max_depth=5):
    """Detecta e parseia colunas string que contêm JSON, depois achata."""
    for _ in range(max_depth):
        str_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        if not str_cols:
            break

        amostra = df.select(*[df[c] for c in str_cols]).filter(
            df[str_cols[0]].isNotNull()
        ).first()
        if not amostra:
            break

        cols_json = []
        for i, c in enumerate(str_cols):
            val = amostra[i]
            if val and isinstance(val, str) and val.strip() and val.strip()[0] in ('{', '['):
                cols_json.append((c, val))

        if not cols_json:
            break

        for c, val in cols_json:
            schema = schema_of_json(val)
            df = df.withColumn(c, from_json(df[c], schema))

        df = flatten_df(df, sep)
    return df

# Para cada arquivo com JSON, parsear, expandir sub-JSONs e achatar
for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"--- {nome}: Sem JSON, mostrando dados diretos ---")
        info["df"].show(5, truncate=False)
        print()
        continue

    df = info["df"]
    for col_json in info["colunas_json"]:
        print(f"--- {nome} | Coluna: {col_json} ---")
        amostra = df.select(col_json).filter(df[col_json].isNotNull()).first()[0]
        json_schema = schema_of_json(amostra)

        df_parsed = df.withColumn("parsed", from_json(df[col_json], json_schema)) \
                      .select("parsed.*")

        # Achatar StructTypes e ArrayTypes
        df_flat = flatten_df(df_parsed)

        colunas_antes = set(df_flat.columns)

        # Parsear sub-JSONs restantes e achatar novamente
        df_flat = parse_nested_json(df_flat)

        colunas_expandidas = [c for c in df_flat.columns if c not in colunas_antes]
        if colunas_expandidas:
            print(f"Colunas JSON aninhadas expandidas: {colunas_expandidas}")

        print(f"Total de colunas: {len(df_flat.columns)}")
        df_flat.printSchema()
        df_flat.show(5, truncate=False)

        info["df_parsed"] = df_flat
    print()

--- Payment_20260207170548.parquet: Sem JSON, mostrando dados diretos ---
+-------------+
|raw_value    |
+-------------+
|QueryResponse|
|time         |
+-------------+


--- Payment_20260325043100.parquet | Coluna: raw_value ---
Colunas JSON aninhadas expandidas: ['CurrencyRef_name', 'CurrencyRef_value', 'CustomerRef_name', 'CustomerRef_value', 'DepositToAccountRef_name', 'DepositToAccountRef_value', 'Line_Amount', 'Line_LinkedTxn_TxnId', 'Line_LinkedTxn_TxnType', 'LinkedTxn_TxnId', 'LinkedTxn_TxnType', 'MetaData_CreateTime', 'MetaData_LastUpdatedTime', 'MetaData_source', 'PaymentMethodRef_name', 'PaymentMethodRef_value']
Total de colunas: 26
root
 |-- CurrencyRef_name: string (nullable = true)
 |-- CurrencyRef_value: string (nullable = true)
 |-- CustomerRef_name: string (nullable = true)
 |-- CustomerRef_value: string (nullable = true)
 |-- DepositToAccountRef_name: string (nullable = true)
 |-- DepositToAccountRef_value: string (nullable = true)
 |-- ExchangeRate: string (nullable

In [5]:
from IPython.display import display
from functools import reduce
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import StringType

# Juntar todos os DataFrames parseados em um só, tratando diferenças de schema e tipos
dfs_parsed = [info["df_parsed"] for info in dataframes.values() if "df_parsed" in info]

if dfs_parsed:
    # Converter todas as colunas para string para evitar conflitos de tipo no union
    dfs_str = []
    for df in dfs_parsed:
        df_str = df.select([col(c).cast(StringType()).alias(c) for c in df.columns])
        dfs_str.append(df_str)

    df_merged = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_str)

    # Identificar e remover colunas que são totalmente nulas
    total = df_merged.count()
    nulls_por_coluna = df_merged.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in df_merged.columns]
    ).first()

    colunas_nulas = [c for c in df_merged.columns if nulls_por_coluna[c] == total]
    if colunas_nulas:
        print(f"Colunas totalmente nulas removidas: {colunas_nulas}")
        df_merged = df_merged.drop(*colunas_nulas)

    print(f"Total de linhas: {total} | Total de colunas: {len(df_merged.columns)}")
    display(df_merged.limit(10).toPandas())
else:
    print("Nenhum DataFrame parseado encontrado.")

Colunas totalmente nulas removidas: ['CurrencyRef', 'CustomerRef', 'DepositToAccountRef', 'Line', 'LinkedTxn', 'MetaData', 'PaymentMethodRef']
Total de linhas: 11 | Total de colunas: 26


,CurrencyRef_name,CurrencyRef_value,CustomerRef_name,CustomerRef_value,DepositToAccountRef_name,DepositToAccountRef_value,ExchangeRate,Id,Line_Amount,Line_LinkedTxn_TxnId,...,PaymentMethodRef_name,PaymentMethodRef_value,PaymentRefNum,PrivateNote,SyncToken,TotalAmt,TxnDate,UnappliedAmt,domain,sparse
0,United States Dollar,USD,Patient 1,cust-001,Operating Account 1,acct-001,1,pay-001,183.75,inv-001,...,Check,paymethod-2,PMT-8001,Collected at desk 1,701,183.75,2026-03-01,6.50,QBO,false
1,United States Dollar,USD,Patient 2,cust-002,Operating Account 1,acct-001,1,pay-002,217.50,inv-002,...,Card,paymethod-3,PMT-8002,Collected at desk 2,702,217.50,2026-03-02,8.00,QBO,false
2,United States Dollar,USD,Patient 3,cust-003,Operating Account 1,acct-001,1,pay-003,251.25,inv-003,...,Cash,paymethod-1,PMT-8003,Collected at desk 3,703,251.25,2026-03-03,9.50,QBO,false
3,United States Dollar,USD,Patient 4,cust-004,Operating Account 1,acct-001,1,pay-004,285.00,inv-004,...,Check,paymethod-2,PMT-8004,Collected at desk 4,704,285.00,2026-03-04,11.00,QBO,false
4,United States Dollar,USD,Patient 5,cust-005,Operating Account 1,acct-001,1,pay-005,318.75,inv-005,...,Card,paymethod-3,PMT-8005,Collected at desk 5,705,318.75,2026-03-05,12.50,QBO,false
5,United States Dollar,USD,Patient 6,cust-006,Operating Account 1,acct-001,1,pay-006,352.50,inv-006,...,Cash,paymethod-1,PMT-8006,Collected at desk 6,706,352.50,2026-03-06,14.00,QBO,false
6,United States Dollar,USD,Patient 7,cust-007,Operating Account 1,acct-001,1,pay-007,386.25,inv-007,...,Check,paymethod-2,PMT-8007,Collected at desk 7,707,386.25,2026-03-07,15.50,QBO,false
7,United States Dollar,USD,Patient 8,cust-008,Operating Account 1,acct-001,1,pay-008,420.00,inv-008,...,Card,paymethod-3,PMT-8008,Collected at desk 8,708,420.00,2026-03-08,17.00,QBO,false
8,United States Dollar,USD,Patient 9,cust-009,Operating Account 1,acct-001,1,pay-009,453.75,inv-009,...,Cash,paymethod-1,PMT-8009,Collected at desk 9,709,453.75,2026-03-09,18.50,QBO,false
9,United States Dollar,USD,Patient 10,cust-010,Operating Account 1,acct-001,1,pay-010,487.50,inv-010,...,Check,paymethod-2,PMT-8010,Collected at desk 10,710,487.50,2026-03-10,20.00,QBO,false


In [6]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
